In [ ]:
-- MigrateX Demo: MSSQL Source Data
-- Database: MSSQL_DEMO_2908
-- Generates ~3.5M rows directly inside SQL Server
-- Run: sqlcmd -S server -U sa -P password -i demo/mssql_setup.sql

IF NOT EXISTS (SELECT * FROM sys.databases WHERE name = 'MSSQL_DEMO_2908')
    CREATE DATABASE MSSQL_DEMO_2908;
GO

USE MSSQL_DEMO_2908;
GO

DROP TABLE IF EXISTS dbo.ORDER_ITEMS;
DROP TABLE IF EXISTS dbo.ORDERS;
DROP TABLE IF EXISTS dbo.PRODUCTS;
DROP TABLE IF EXISTS dbo.CUSTOMERS;
DROP TABLE IF EXISTS dbo._LOOKUP_FIRST;
DROP TABLE IF EXISTS dbo._LOOKUP_LAST;
DROP TABLE IF EXISTS dbo._LOOKUP_CITY;
DROP TABLE IF EXISTS dbo._LOOKUP_CAT;
DROP TABLE IF EXISTS dbo._LOOKUP_STATUS;
GO

-- Lookup tables (avoids CHOOSE() NULL bugs)
CREATE TABLE dbo._LOOKUP_FIRST (id INT PRIMARY KEY, val NVARCHAR(20));
INSERT INTO dbo._LOOKUP_FIRST VALUES
(1,'James'),(2,'Mary'),(3,'John'),(4,'Patricia'),(5,'Robert'),
(6,'Jennifer'),(7,'Michael'),(8,'Linda'),(9,'David'),(10,'Elizabeth'),
(11,'William'),(12,'Barbara'),(13,'Richard'),(14,'Susan'),(15,'Joseph'),
(16,'Jessica'),(17,'Thomas'),(18,'Sarah'),(19,'Charles'),(20,'Karen');

CREATE TABLE dbo._LOOKUP_LAST (id INT PRIMARY KEY, val NVARCHAR(20));
INSERT INTO dbo._LOOKUP_LAST VALUES
(1,'Smith'),(2,'Johnson'),(3,'Williams'),(4,'Brown'),(5,'Jones'),
(6,'Garcia'),(7,'Miller'),(8,'Davis'),(9,'Rodriguez'),(10,'Martinez'),
(11,'Hernandez'),(12,'Lopez'),(13,'Gonzalez'),(14,'Wilson'),(15,'Anderson'),
(16,'Thomas'),(17,'Taylor'),(18,'Moore'),(19,'Jackson'),(20,'Martin');

CREATE TABLE dbo._LOOKUP_CITY (id INT PRIMARY KEY, val NVARCHAR(30), st NVARCHAR(5));
INSERT INTO dbo._LOOKUP_CITY VALUES
(1,'New York','NY'),(2,'Los Angeles','CA'),(3,'Chicago','IL'),(4,'Houston','TX'),
(5,'Phoenix','AZ'),(6,'Philadelphia','PA'),(7,'San Antonio','TX'),
(8,'San Diego','CA'),(9,'Dallas','TX'),(10,'San Jose','CA');

CREATE TABLE dbo._LOOKUP_CAT (id INT PRIMARY KEY, val NVARCHAR(20));
INSERT INTO dbo._LOOKUP_CAT VALUES
(1,'Electronics'),(2,'Clothing'),(3,'Home'),(4,'Sports'),
(5,'Books'),(6,'Toys'),(7,'Food'),(8,'Auto');

CREATE TABLE dbo._LOOKUP_STATUS (id INT PRIMARY KEY, val NVARCHAR(20));
INSERT INTO dbo._LOOKUP_STATUS VALUES
(1,'completed'),(2,'completed'),(3,'completed'),(4,'shipped'),(5,'processing'),(6,'cancelled');
GO

-- Main tables
CREATE TABLE dbo.CUSTOMERS (
    CUSTOMER_ID INT PRIMARY KEY,
    FIRST_NAME NVARCHAR(50) NOT NULL,
    LAST_NAME NVARCHAR(50) NOT NULL,
    EMAIL NVARCHAR(120),
    CITY NVARCHAR(60),
    STATE NVARCHAR(10),
    COUNTRY NVARCHAR(10),
    CREATED_AT DATETIME2 NOT NULL,
    UPDATED_AT DATETIME2 NOT NULL
);
CREATE TABLE dbo.PRODUCTS (
    PRODUCT_ID INT PRIMARY KEY,
    PRODUCT_NAME NVARCHAR(100) NOT NULL,
    CATEGORY NVARCHAR(30),
    PRICE MONEY,
    STOCK_QTY INT,
    CREATED_AT DATETIME2 NOT NULL,
    UPDATED_AT DATETIME2 NOT NULL
);
CREATE TABLE dbo.ORDERS (
    ORDER_ID INT PRIMARY KEY,
    CUSTOMER_ID INT NOT NULL,
    ORDER_DATE DATETIME2 NOT NULL,
    STATUS NVARCHAR(20),
    TOTAL_AMOUNT MONEY,
    SHIP_CITY NVARCHAR(60),
    UPDATED_AT DATETIME2 NOT NULL
);
CREATE TABLE dbo.ORDER_ITEMS (
    ITEM_ID BIGINT PRIMARY KEY,
    ORDER_ID INT NOT NULL,
    PRODUCT_ID INT NOT NULL,
    QUANTITY INT,
    UNIT_PRICE MONEY,
    CREATED_AT DATETIME2 NOT NULL
);
GO

-- Helper: deterministic random datetime
CREATE OR ALTER FUNCTION dbo.fn_rand_date(@seed INT)
RETURNS DATETIME2 AS
BEGIN
    RETURN DATEADD(SECOND,
        ABS(CHECKSUM(HASHBYTES('MD5', CAST(@seed AS VARCHAR(20)))) % 83980800),
        '2024-01-01');
END;
GO

-- Products (10K)
PRINT 'Generating Products...';
;WITH nums AS (
    SELECT TOP 10000 ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS n
    FROM sys.all_objects a CROSS JOIN sys.all_objects b
)
INSERT INTO dbo.PRODUCTS
SELECT n,
    CONCAT(c1.val, ' ', c2.val, ' ', n),
    c3.val,
    CAST(5 + RAND(CHECKSUM(NEWID())) * 995 AS DECIMAL(10,2)),
    ABS(CHECKSUM(NEWID()) % 10000),
    dbo.fn_rand_date(n),
    dbo.fn_rand_date(n + 10000)
FROM nums
CROSS APPLY (SELECT val FROM dbo._LOOKUP_CAT WHERE id = 1 + ABS(CHECKSUM(NEWID()) % 8)) c1  -- reuse as adj
CROSS APPLY (SELECT val FROM dbo._LOOKUP_CAT WHERE id = 1 + ABS(CHECKSUM(NEWID()) % 8)) c2  -- reuse as noun
CROSS APPLY (SELECT val FROM dbo._LOOKUP_CAT WHERE id = 1 + ABS(CHECKSUM(NEWID()) % 8)) c3;
PRINT 'Products: 10,000 done';
GO

-- Customers (500K) in batches of 50K
PRINT 'Generating Customers...';
DECLARE @batch INT = 0;
WHILE @batch < 10
BEGIN
    ;WITH nums AS (
        SELECT TOP 50000 ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS n
        FROM sys.all_objects a CROSS JOIN sys.all_objects b
    )
    INSERT INTO dbo.CUSTOMERS
    SELECT
        @batch * 50000 + n,
        fn.val,
        ln.val,
        CONCAT(LOWER(LEFT(fn.val,4)), '.', LOWER(LEFT(ln.val,4)), @batch * 50000 + n, '@example.com'),
        ct.val,
        ct.st,
        CASE ABS(CHECKSUM(NEWID()) % 5) WHEN 0 THEN 'CA' WHEN 1 THEN 'UK' ELSE 'US' END,
        dbo.fn_rand_date(@batch * 50000 + n),
        dbo.fn_rand_date(@batch * 50000 + n + 500000)
    FROM nums
    CROSS APPLY (SELECT val FROM dbo._LOOKUP_FIRST WHERE id = 1 + ABS(CHECKSUM(NEWID()) % 20)) fn
    CROSS APPLY (SELECT val FROM dbo._LOOKUP_LAST WHERE id = 1 + ABS(CHECKSUM(NEWID()) % 20)) ln
    CROSS APPLY (SELECT val, st FROM dbo._LOOKUP_CITY WHERE id = 1 + ABS(CHECKSUM(NEWID()) % 10)) ct;

    SET @batch = @batch + 1;
    PRINT CONCAT('Customers batch ', @batch, '/10 done');
END;
GO

-- Orders (1M) in batches of 50K
PRINT 'Generating Orders...';
DECLARE @batch INT = 0;
WHILE @batch < 20
BEGIN
    ;WITH nums AS (
        SELECT TOP 50000 ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS n
        FROM sys.all_objects a CROSS JOIN sys.all_objects b
    )
    INSERT INTO dbo.ORDERS
    SELECT
        @batch * 50000 + n,
        1 + ABS(CHECKSUM(NEWID()) % 500000),
        dbo.fn_rand_date(@batch * 50000 + n),
        s.val,
        CAST(10 + RAND(CHECKSUM(NEWID())) * 2490 AS DECIMAL(12,2)),
        ct.val,
        dbo.fn_rand_date(@batch * 50000 + n + 1000000)
    FROM nums
    CROSS APPLY (SELECT val FROM dbo._LOOKUP_STATUS WHERE id = 1 + ABS(CHECKSUM(NEWID()) % 6)) s
    CROSS APPLY (SELECT val FROM dbo._LOOKUP_CITY WHERE id = 1 + ABS(CHECKSUM(NEWID()) % 10)) ct;

    SET @batch = @batch + 1;
    PRINT CONCAT('Orders batch ', @batch, '/20 done');
END;
GO

-- Order Items (2M) in batches of 50K
PRINT 'Generating Order Items...';
DECLARE @batch INT = 0;
WHILE @batch < 40
BEGIN
    ;WITH nums AS (
        SELECT TOP 50000 ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS n
        FROM sys.all_objects a CROSS JOIN sys.all_objects b
    )
    INSERT INTO dbo.ORDER_ITEMS
    SELECT
        @batch * 50000 + n,
        1 + ABS(CHECKSUM(NEWID()) % 1000000),
        1 + ABS(CHECKSUM(NEWID()) % 10000),
        1 + ABS(CHECKSUM(NEWID()) % 10),
        CAST(5 + RAND(CHECKSUM(NEWID())) * 495 AS DECIMAL(10,2)),
        dbo.fn_rand_date(@batch * 50000 + n)
    FROM nums;

    SET @batch = @batch + 1;
    PRINT CONCAT('Order Items batch ', @batch, '/40 done');
END;
GO

-- Cleanup lookup tables
DROP TABLE dbo._LOOKUP_FIRST;
DROP TABLE dbo._LOOKUP_LAST;
DROP TABLE dbo._LOOKUP_CITY;
DROP TABLE dbo._LOOKUP_CAT;
DROP TABLE dbo._LOOKUP_STATUS;
GO

PRINT 'MSSQL DEMO DATA GENERATION COMPLETE';
GO
